# Variables aleatorias

Una **variable aleatoria** es una función $X: \Omega \to \mathbb{R}$ que le
asigna un número a cada resultado del experimento. Esa traducción permite
tres movimientos clave:

- Hablar de la **distribución** de los valores (PMF para discretas, PDF para
  continuas).
- Resumir la distribución con **esperanza** y **varianza** sin enumerar el
  espacio muestral.
- Pasar del razonamiento por enumeración del capítulo anterior al razonamiento
  por **densidades** y **áreas**.

Las situaciones que veníamos siguiendo encuentran ahora un modelo natural: las
esperas de la clínica se describen bien con una **Exponencial**, el número de
«sí» en una tanda de la encuesta con una **Binomial**, y los defectos por turno
de la línea de producción con una **Poisson**.


## Preguntas de inicio

1. ¿Cuál es la probabilidad de que la espera supere los 5 minutos si
   modelamos como Exponencial?
2. En 50 inspecciones, ¿cuál es la probabilidad de exactamente 4 defectos?
3. ¿Qué porción de personas tiene altura entre 165 y 180 cm si la altura
   sigue una Normal?
4. ¿Cómo invertimos la pregunta: a qué altura corresponde el percentil 90?


## Importaciones

In [ ]:
from core import (
    BinomialParams,
    ExponentialParams,
    NormalParams,
    PoissonParams,
    Settings,
)
from distributions import (
    DensityGridInput,
    MomentsInput,
    ProbabilityMassInput,
    QuantileInput,
    TailProbabilityInput,
    compute_numeric_moments,
    evaluate_density_grid,
    evaluate_probability_mass,
    make_binomial,
    make_exponential,
    make_normal,
    make_poisson,
    quantile_of_continuous,
    tail_probability_of_continuous,
)
from exercises import NumericAnswerInput, verify_numeric_answer
from symbolic import (
    compute_binomial_moments,
    compute_poisson_moments,
    standardize_normal,
)
from visualization import (
    DensityChartInput,
    ProbabilityMassChartInput,
    chart_density,
    chart_probability_mass,
)
from widgets import (
    ContinuousDistributionExplorerInput,
    DiscreteDistributionExplorerInput,
    build_continuous_distribution_explorer,
    build_discrete_distribution_explorer,
)

In [ ]:
settings = Settings()

## Contar defectos con una Binomial

Inspeccionamos $n = 50$ piezas, y cada una tiene probabilidad $p = 0{,}05$ de
ser defectuosa. La cantidad $X$ de piezas defectuosas es
$X \sim \text{Bin}(50, 0{,}05)$.

Su PMF (función de probabilidad puntual) viene dada por:

$$ P(X = k) = \binom{n}{k}\,p^{k}\,(1 - p)^{n - k},\quad k = 0, 1, \dots, n \tag{3.1} $$


In [ ]:
factory_distribution = make_binomial(BinomialParams(trials=50, success_probability=0.05))
factory_mass_input = ProbabilityMassInput(
    distribution=factory_distribution,
    lower_outcome=0,
    upper_outcome=12,
)
factory_mass = evaluate_probability_mass(factory_mass_input)

factory_chart_input = ProbabilityMassChartInput(
    probability_mass=factory_mass,
    title="Defectos en 50 inspecciones — Bin(50, 0.05)",
    settings=settings,
)
chart_probability_mass(factory_chart_input)

La esperanza y la varianza tienen, en la Binomial, formas particularmente
limpias: $E[X] = np$ y $\mathrm{Var}(X) = np(1-p)$. Conviene verlas escritas
una vez para no recalcularlas en cada situación nueva:


In [ ]:
factory_moments_symbolic = compute_binomial_moments(BinomialParams(trials=50, success_probability=0.05))
factory_moments_symbolic

In [ ]:
factory_moments_input = MomentsInput(distribution=factory_distribution)
factory_moments_numeric = compute_numeric_moments(factory_moments_input)
factory_moments_numeric

## Alturas con la Normal

Antes de meternos con tiempos cambiamos completamente de escenario para
presentar la Normal en un caso clásico: la altura adulta de una población.
Tomamos $X \sim \mathcal{N}(170, 8^2)$.

**Paso 1.** La PDF de una Normal con parámetros $\mu$ y $\sigma$ es:

$$ f_X(x) = \frac{1}{\sigma\sqrt{2\pi}}\,\exp\!\left(-\frac{(x - \mu)^2}{2\sigma^2}\right) \tag{3.2} $$

**Paso 2.** La probabilidad de un intervalo se obtiene integrando (3.2):

$$ P(a \le X \le b) = \int_a^b f_X(x)\,dx \tag{3.3} $$


In [ ]:
heights_distribution = make_normal(NormalParams(mean=170.0, standard_deviation=8.0))
heights_density_input = DensityGridInput(
    distribution=heights_distribution,
    settings=settings,
)
heights_density = evaluate_density_grid(heights_density_input)

heights_chart_input = DensityChartInput(
    density_grid=heights_density,
    title="Altura adulta — Normal(170, 8²)",
    settings=settings,
)
chart_density(heights_chart_input)

**Paso 3.** Aplicamos (3.3) con $a = 165$ y $b = 180$:


In [ ]:
heights_interval_input = TailProbabilityInput(
    distribution=heights_distribution,
    lower_bound=165.0,
    upper_bound=180.0,
)
heights_interval_probability = tail_probability_of_continuous(heights_interval_input)
heights_interval_probability

## Estandarización: el puente con el Capítulo 1

El $z$-score de la ecuación (1.5) **es la misma transformación** que ahora
vamos a usar para llevar cualquier Normal a la Normal estándar. **Paso 1:**
partimos de $X \sim \mathcal{N}(\mu, \sigma^2)$. **Paso 2:** definimos
$Z$ por:

$$ Z = \frac{X - \mu}{\sigma} \tag{3.4} $$

Y se demuestra que $Z \sim \mathcal{N}(0, 1)$. La diferencia con (1.5) es
que ahora **conocemos** la distribución del resultado, no solo su muestra.


In [ ]:
standardize_normal(NormalParams(mean=170.0, standard_deviation=8.0)).formula

## Cuantiles: invertir la pregunta

En lugar de fijar $x$ y leer probabilidad, fijamos probabilidad y leemos $x$.
El **percentil $\alpha$** se define como el valor $x_\alpha$ tal que
$P(X \le x_\alpha) = \alpha$. Esto es la inversa del CDF.

$$ x_\alpha = F_X^{-1}(\alpha) \tag{3.5} $$


In [ ]:
heights_quantile_input = QuantileInput(
    distribution=heights_distribution,
    probability=0.90,
)
heights_percentile_ninety = quantile_of_continuous(heights_quantile_input)
heights_percentile_ninety

## El tiempo de espera como Exponencial

El tiempo $T$ entre eventos sin memoria (la próxima llegada, la próxima
atención) suele modelarse Exponencial con tasa $\lambda$. Su densidad es:

$$ f_T(t) = \lambda\,e^{-\lambda t},\quad t \ge 0 \tag{3.6} $$

Tomemos $\lambda = 0{,}25$ esperas por minuto, lo que da una espera promedio
$E[T] = 1/\lambda = 4$ minutos (consistente con la muestra del Capítulo 1).
**Pregunta de inicio 1:** la probabilidad de superar 5 minutos.

**Paso 1.** Aplicamos (3.3) con $a = 5$, $b = \infty$ y la densidad (3.6).
**Paso 2.** El resultado conocido es $P(T > t) = e^{-\lambda t}$, así que:

$$ P(T > 5) = e^{-0{,}25 \cdot 5} = e^{-1{,}25} \approx 0{,}287 $$


In [ ]:
clinic_distribution = make_exponential(ExponentialParams(rate=0.25))
clinic_tail_input = TailProbabilityInput(
    distribution=clinic_distribution,
    lower_bound=5.0,
)
clinic_tail_probability = tail_probability_of_continuous(clinic_tail_input)
clinic_tail_probability

## Exploración interactiva — distribuciones continuas

Cambiamos familia, parámetros y un intervalo $[x_{\min}, x_{\max}]$. La
probabilidad se actualiza en vivo. Ideal para sentir cómo la cola crece o se
encoge cuando movemos $\sigma$ o la tasa.


In [ ]:
continuous_explorer_input = ContinuousDistributionExplorerInput(
    settings=settings,
)
build_continuous_distribution_explorer(continuous_explorer_input)

## Cuántas personas dicen «sí»

Una encuesta toma una tanda de $n = 30$ personas. Si la verdadera proporción
de «sí» en la población es $p = 0{,}55$, la cantidad $Y$ de respuestas «sí»
en la tanda sigue $Y \sim \text{Bin}(30, 0{,}55)$. Reusamos (3.1):


In [ ]:
poll_distribution = make_binomial(BinomialParams(trials=30, success_probability=0.55))
poll_mass_input = ProbabilityMassInput(
    distribution=poll_distribution,
    lower_outcome=0,
    upper_outcome=30,
)
poll_mass = evaluate_probability_mass(poll_mass_input)

poll_chart_input = ProbabilityMassChartInput(
    probability_mass=poll_mass,
    title="Encuesta de 30 personas — Bin(30, 0.55)",
    settings=settings,
)
chart_probability_mass(poll_chart_input)

## Poisson como límite

Cuando $n$ es grande y $p$ es chico (con $\lambda = np$ moderado), la
Binomial converge a la Poisson. La PMF Poisson es:

$$ P(K = k) = \frac{\lambda^{k}\,e^{-\lambda}}{k!} \tag{3.7} $$

Para los defectos por turno con $\lambda = 4$:


In [ ]:
poisson_distribution = make_poisson(PoissonParams(rate=4.0))
poisson_mass_input = ProbabilityMassInput(
    distribution=poisson_distribution,
    lower_outcome=0,
    upper_outcome=12,
)
poisson_mass = evaluate_probability_mass(poisson_mass_input)

poisson_chart_input = ProbabilityMassChartInput(
    probability_mass=poisson_mass,
    title="Defectos por turno — Poisson(4)",
    settings=settings,
)
chart_probability_mass(poisson_chart_input)

## Exploración interactiva — discretas

Pasamos de Binomial a Poisson moviendo el dropdown. Cuando $n$ es grande y
$p$ es chico la PMF Binomial casi se superpone con la Poisson de tasa $np$.


In [ ]:
discrete_explorer_input = DiscreteDistributionExplorerInput(settings=settings)
build_discrete_distribution_explorer(discrete_explorer_input)

## Ejercicio 1 — Estandarizar y leer en la Normal estándar

$X \sim \mathcal{N}(170, 8^2)$. Calculá $P(X \le 178)$.

**Idea.** Aplicar (3.4) y reducir el problema a $P(Z \le 1)$.

$$ \begin{aligned}
z &= \frac{178 - 170}{8} \\[4pt]
  &= 1 \\[4pt]
P(X \le 178) &= P(Z \le 1) \approx 0{,}8413
\end{aligned} $$


In [ ]:
exercise_tail_input = TailProbabilityInput(
    distribution=heights_distribution,
    upper_bound=178.0,
)
expected_probability = tail_probability_of_continuous(exercise_tail_input).probability

student_answer_probability = 0.8413
verify_input = NumericAnswerInput(
    student_answer=student_answer_probability,
    expected_answer=expected_probability,
    absolute_tolerance=1e-3,
)
verify_numeric_answer(verify_input)

## Ejercicio 2 — Esperanza de una Poisson

Si $K \sim \text{Poisson}(3{,}5)$, ¿cuál es $E[K]$?

**Idea.** Para Poisson, $E[K] = \lambda = 3{,}5$. La Poisson **es su tasa**.


In [ ]:
expected_expectation = compute_poisson_moments(PoissonParams(rate=3.5)).expectation

student_answer_expectation = 3.5
verify_input = NumericAnswerInput(
    student_answer=student_answer_expectation,
    expected_answer=float(expected_expectation),
)
verify_numeric_answer(verify_input)

## Síntesis y respuestas

1. **¿Probabilidad de espera mayor a 5 minutos?** Modelando con Exponencial
   de tasa $0{,}25$, la cola exponencial $P(T > 5) = e^{-1{,}25}$ deja la
   chance en torno al 28,7%.
2. **¿Probabilidad de exactamente 4 defectos en 50 inspecciones?** La PMF
   Binomial (3.1) con $n = 50$, $p = 0{,}05$ y $k = 4$ da el valor exacto;
   alcanza con leerlo en la tabla de masa graficada al inicio.
3. **¿Porción con altura entre 165 y 180 cm?** Integrando la densidad (3.2)
   entre esos valores queda alrededor del 63% de la población — la porción
   central de la nube de alturas.
4. **¿A qué altura corresponde el percentil 90?** Por (3.5),
   $F_X^{-1}(0{,}90) \approx 180$ cm. Diez de cada cien adultos en esa
   población superan esa marca.


Cada distribución de este capítulo describe **una sola observación**: el
tiempo de un paciente, la respuesta de una persona, el conteo de un turno.
Pero ningún día de la clínica se decide con un solo paciente, ni una encuesta
se publica con una sola respuesta, ni una jornada de la línea se resume en un
único conteo. Lo que importa, casi siempre, es el **promedio** de muchas
observaciones — y ahí aparecen preguntas que con lo de hasta acá no se pueden
contestar: ¿cómo se distribuye el promedio de treinta esperas?, ¿por qué la
encuesta gana confiabilidad a medida que crece la muestra?, ¿por qué tantos
fenómenos terminan pareciendo Normales aunque la fuente no lo sea? Las tres
respuestas comparten el mismo teorema y van a ocupar buena parte del recorrido
que sigue.
